This notebook computes the walking times between pairs of grocery stores.

In [1]:
# Initialization
import numpy as np
import geopandas as gpd
from shapely.geometry import Point, LineString, Polygon
from shapely.geometry import MultiPoint
import networkx as nx
import osmnx as ox

In [2]:
# Location of data
datadir = '../Data/'

# On AKB's computer while testing
#datadir = '../../tda-resources-dallas/Dallas/'

In [ ]:
#Loading street network

# Our set of grocery stores has points in Collin County 
# (It is City of Dallas, which actually crosses the county line (while also not including non-COD
#     places in Dallas Co)
# This can cause errors when attempting to find path between nodes. 

use_bbox_for_grid = False

if use_bbox_for_grid:
    # To include the southern piece of Collin Co, use bbox
    # bbox = [-97.1,32.5,-96.45,33.1]
    # bbox = [-97.8080, 32.7725, -96.7900, 32.7915]
    coords_list = [-96.808, 32.780, -96.788, 32.807]
    G = ox.graph_from_bbox(coords_list,network_type='drive', simplify=False)
   # G_df = ox.geocode.geocode_to_(bbox,network_type='walk', simplify=False)
else: 
    place = 'Dallas, Texas, United States' #!! replace with desired city here
    #G = ox.graph_from_place(place,network_type='walk', buffer_dist = 5000, simplify=False)
    G = ox.graph_from_place(place,network_type='drive', simplify=False)


# four_tuples = [
#     (-96.808, 32.780),
#     (-96.808, 32.807),
#     (-96.788, 32.807),# Top-Left (Northwest)
#     (-96.788, 32.780) # Top-Right (Northeast)
#      # Bottom-Right (Southeast)
#       # Bottom-Left (Southwest)
# ]

# # Create the bounding area polygon
# geo_area = Polygon(four_tuples)

# geo_area
# # print(f"Is this area a valid shape? {geo_area.is_valid}")

# # G.draw()

In [5]:
groc_sites = gpd.read_file(datadir+'geo_export_872fcb6c-fbde-4264-ae77-8858a604ed0e.shp') # load grocery store sites

# # groc_sites.head()
# groc_sites_select = groc_sites[groc_sites['city'] == 'Dallas']
# groc_sites_select.head()

# groc_sites_select['intersects'] = groc_sites_select.within(geo_area)
# groc_sites_select_inner = groc_sites_select[groc_sites_select['intersects'] == True]
# groc_sites_select_inner.head()

In [6]:
# For each site, find closest "node" in the street graph
# nodes=ox.distance.nearest_nodes(G,groc_sites_select_inner['geometry'].x,groc_sites_select_inner['geometry'].y)
nodes=ox.distance.nearest_nodes(G,groc_sites['geometry'].x,groc_sites['geometry'].y)

In [7]:
nodes

array([ 5785655409,  5831032558,  5882491639,    81821079,    81880125,
         150215996,    82128101,  6738009230,    81786353,  6740141283,
          82109873,  7680551612,  4192359964,    81778090,    81759562,
          82105376,    81857280,  7589237497,  9647750201, 13533056946,
        6952472553,    82218849,    81715857,  5462598528,    81915167,
       11634211954, 13826579978,  6749327519,  4470687050,  9478439089,
        6377767334,  9895056923,  5828281600, 10867918147,  6749720983,
        5817095820,  5797819126,    82225370,  6958795484,    81652939,
          81926523,    82110256,  8178374162,  5789642101,    81858222,
        3708315412,  9874385500,    82111600,  2903091553,  5400597668,
        5409365805, 11464423691, 10805806992,  2903091550,  7953358834,
          81697675,   315089918,    82001918,    81973067,    81561310,
        3736097493, 10633777282, 10840309426,  8255924489,  5389653243,
        9531936617,  8980776513,  5802650872,    81975354, 13140

In [8]:
# Matrix for distance calculations.
drive=np.full((len(groc_sites),len(groc_sites)),0)

In [23]:
#List pairs of indices for distance computation
#
# Why? If parallel processing, this allows individual computations to be farmed out 
# in arbitrary order
pairs_list = []

# Is the matrix expected to be symmetric? If so, 
# do not calculate d(i,j) and d(j,i) separately.
dist_is_symmetric = True

for i in range(len(nodes)):
    for j in range(i+1,len(nodes)):
        pairs_list.append((i,j))

    if not dist_is_symmetric:
        for j in range(0,i):
            pairs_list.append((i,j))

In [11]:
#pairs_list

In [26]:
# compute walk distance (in meters) for each pair of resource sites
mylist = []

for p1,p2 in pairs_list:
    # if (p2 == p1 + 1):
        # Just to get a feel how quickly this is going
        #print(p1,p2)
    try: 
        mylist.append(nx.shortest_path_length(G,nodes[p1],nodes[p2],weight='length'))
        #print(nx.shortest_path_length(G,nodes[p1],nodes[p2],weight='length'))
    except Exception as e: 
        print(f"Error for {nodes[p1]} and {nodes[p2]}: {e}")
        mylist.append(0)

Error for 5785655409 and 8628662412: Node 8628662412 not reachable from 5785655409
Error for 5785655409 and 9861766727: Node 9861766727 not reachable from 5785655409
Error for 5831032558 and 8628662412: Node 8628662412 not reachable from 5831032558
Error for 5831032558 and 9861766727: Node 9861766727 not reachable from 5831032558
Error for 5882491639 and 8628662412: Node 8628662412 not reachable from 5882491639
Error for 5882491639 and 9861766727: Node 9861766727 not reachable from 5882491639
Error for 81821079 and 8628662412: Node 8628662412 not reachable from 81821079
Error for 81821079 and 9861766727: Node 9861766727 not reachable from 81821079
Error for 81880125 and 8628662412: Node 8628662412 not reachable from 81880125
Error for 81880125 and 9861766727: Node 9861766727 not reachable from 81880125
Error for 150215996 and 8628662412: Node 8628662412 not reachable from 150215996
Error for 150215996 and 9861766727: Node 9861766727 not reachable from 150215996
Error for 82128101 and 8

KeyboardInterrupt: 

In [20]:
# print(len(pairs_list))

9453


In [22]:
# print(len(mylist))

137


In [17]:
# mylist

[np.float64(16291.487486000104),
 np.float64(12494.090183666973),
 np.float64(6873.131880811609),
 np.float64(6813.396387360747),
 np.float64(26558.702284045372),
 np.float64(19652.599797384904),
 np.float64(26657.647414685154),
 np.float64(23154.493421495827),
 np.float64(11824.630587024294),
 np.float64(12152.026839886983),
 np.float64(3176.543401931971),
 np.float64(15462.242419469887),
 np.float64(11045.790992731481),
 np.float64(10658.92956601893),
 np.float64(16849.050693971694),
 np.float64(16023.628249853036),
 np.float64(9336.353928652736),
 np.float64(19848.26211366841),
 np.float64(24413.564391308777),
 np.float64(25075.53543591931),
 np.float64(19542.23936423931),
 np.float64(32686.614589474266),
 np.float64(17998.623084723222),
 np.float64(2009.1605197202412),
 np.float64(10242.264321372975),
 np.float64(10926.55307519734),
 np.float64(20900.931333293785),
 np.float64(8438.714946634534),
 np.float64(16829.940758967205),
 np.float64(12083.202224379778),
 np.float64(16187.14

In [18]:
#reshape into square array
#pairs list is 9453
#my list is 137 in length
for i in range(len(pairs_list)):
    drive[pairs_list[i][0],pairs_list[i][1]]=mylist[i]
    if dist_is_symmetric:
        drive[pairs_list[i][1],pairs_list[i][0]]=mylist[i]

IndexError: list index out of range

In [35]:
dal_drive_distance_meters = drive
# dal_drive_distance_minutes = dal_drive_distance_seconds/60

In [36]:
#walk_speed = 1.42 # meters/sec

In [37]:
# dal_drive_distance_seconds = dal_drive_distance_meters/walk_speed #compute matrix of walk TIMES

In [38]:
dal_drive_distance_meters

array([[   0,  745,  854, 1292, 1236],
       [ 745,    0, 1027, 1464, 1133],
       [ 854, 1027,    0,  623, 2452],
       [1292, 1464,  623,    0, 2460],
       [1236, 1133, 2452, 2460,    0]])

In [39]:
np.savez('dal_drive_distance_data', dal_drive_distance_meters = dal_drive_distance_meters)